# agent_framework smoke test

Umbrella distribution that re-exports `agent_framework_core[all]` (Microsoft Agent Framework).

In [ ]:
# Example 1: Inspect the top-level public API surface.
import agent_framework

public_names = [n for n in dir(agent_framework) if not n.startswith('_')]
print(f"Number of public names exported: {len(public_names)}")
print(f"First 20 public names: {public_names[:20]}")

# The umbrella re-exports agent_framework_core; spot-check that common identifiers exist
assert len(public_names) > 0, "agent_framework should re-export some public API from core"

In [ ]:
# Example 2: Message / Role construction (core data types).
# agent_framework >= 1.10 renamed ChatMessage -> Message; roles are plain string
# literals ("system" | "user" | "assistant" | "tool") per RoleLiteral, not an enum.
import typing
from agent_framework import Message, RoleLiteral

msg = Message(role="user", contents=["Hello, world!"])
print(f"role = {msg.role}")
print(f"text = {msg.text}")
print(f"type  = {type(msg).__name__}")

assert msg.role == "user"
assert "user" in typing.get_args(RoleLiteral)
assert msg.text == "Hello, world!"

In [ ]:
# Example 3: FunctionTool / @tool decorator (tool definition).
# agent_framework >= 1.10 renamed the ai_function decorator to `tool`; it wraps
# the callable in a FunctionTool carrying name/description/JSON-schema parameters.
from agent_framework import FunctionTool, tool

@tool
def add(a: int, b: int) -> int:
    """Return the sum of two integers."""
    return a + b

print(f"add.name = {add.name}")
print(f"add.description = {add.description}")
print(f"type(add) = {type(add).__name__}")
print(f"parameters = {add.parameters()}")

# The decorator turns `add` into a tool object wrapping the original function
assert isinstance(add, FunctionTool)
assert add.name == "add"
assert add.func(2, 3) == 5

In [ ]:
# Example 4: Importable agent classes / abstractions (no remote calls).
import agent_framework

# Pull a handful of headline names defensively — they may or may not exist
# (>= 1.10 surface: Agent/AgentSession/AgentResponse replaced the old
# ChatAgent/AgentThread/AgentRunResponse names).
candidate_names = [
    "Agent", "AgentSession", "AgentResponse", "BaseAgent",
    "WorkflowBuilder", "Workflow", "Executor",
]
found = {n: getattr(agent_framework, n, None) for n in candidate_names}
for n, obj in found.items():
    status = type(obj).__name__ if obj is not None else "MISSING"
    print(f"  {n}: {status}")

# At least one of these abstractions should be present in any meaningful build
assert any(v is not None for v in found.values()), "no headline agent classes found"

In [ ]:
# Example 5: Exception types + version sanity check.
import agent_framework

exc_names = [n for n in dir(agent_framework) if n.endswith('Error') or n.endswith('Exception')]
print(f"Exception-shaped names: {exc_names[:10]}")

version = getattr(agent_framework, '__version__', None)
print(f"version = {version}")

# Final pass signal
assert agent_framework is not None
print("\nagent_framework smoke test: PASS")